# Telepati 8.0: AI AgriData Intelligence Race
## Deteksi Penyakit Daun Padi Multi-Class (11 Kelas Kanonikal)
---
### Bagian A — Setup & Preprocessing
Instalasi dependensi, unduh dataset, audit anotasi COCO, konversi ke format YOLO, dan verifikasi distribusi kelas.


In [ ]:
%pip install -q gdown matplotlib pandas numpy pillow "ultralytics==8.4.163"

In [ ]:
!gdown "1CAPmwVmYW6499Ic1wx8TFptLV8beDZcX" -O dataset.zip

In [ ]:
!unzip -q -o dataset.zip -d dataset/


## A.1 Import Library & Kunci Determinisme
Mengimpor modul penting serta mengunci determinisme 4-lapis (Python, NumPy, PyTorch CPU, CUDA, dan CuDNN) untuk kepatuhan Audit Run juri.


In [ ]:
import json
import os
import random
import shutil
import time
import zipfile
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


def set_seed(seed: int = 42) -> None:
    """Kunci seluruh sumber keacakan: Python, NumPy, PyTorch CPU/CUDA, cuDNN, dan cuBLAS."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    try:
        import torch

        torch.manual_seed(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


set_seed(42)


## A.2 Konfigurasi Global (`Config`)
Konfigurasi jalur path hibrida (Kaggle vs Lokal), pemetaan 11 kelas kanonikal, serta hyperparameter pelatihan model from scratch.


In [ ]:
def _pilih_device():
    """GPU tunggal (indeks 0) bila ada, selain itu CPU.

    Multi-GPU (DDP) sengaja TIDAK dipakai: Ultralytics menjalankan DDP di proses baru sehingga patch
    BiFPN/EIoU di notebook tidak ikut terbawa, dan hasil menjadi bergantung pada jumlah GPU.
    """
    try:
        import torch

        return 0 if torch.cuda.is_available() else "cpu"
    except ImportError:
        return "cpu"


@dataclass
class Config:
    # --- Deteksi Lingkungan Kerja (Kaggle vs Lokal) ---
    is_kaggle: bool = field(default_factory=lambda: Path("/kaggle/working").exists())

    # --- Path dataset & output ---
    dataset_root: Path = field(
        default_factory=lambda: (
            Path(r"/kaggle/working/dataset/Telepati 8.0 Datasets")
            if Path("/kaggle/working").exists()
            else Path.cwd() / "dataset" / "Telepati 8.0 Datasets"
        )
    )
    yolo_out: Path = field(
        default_factory=lambda: (
            Path(r"/kaggle/working/train")
            if Path("/kaggle/working").exists()
            else Path.cwd() / "train"
        )
    )
    coco_annotation_file: str = "_annotations.coco.json"

    # --- Split dataset ---
    coco_splits: List[str] = field(default_factory=lambda: ["train", "valid", "test"])
    split_mapping: Dict[str, str] = field(
        default_factory=lambda: {"train": "train", "valid": "val", "test": "test"}
    )

    # --- Reprodusibilitas ---
    seed: int = 42

    # --- Mapping & daftar kelas (11 Kelas Kanonikal Resmi) ---
    class_mapping: Dict[str, str] = field(
        default_factory=lambda: {
            "Bacterial leaf blight": "bacterial_leaf_blight",
            "Bacterial panicle Blight": "bacterial_panicle_blight",
            "Blast": "blast",
            "Leaf blast": "blast",
            "Infected Blast": "blast",
            "Brown spot": "brown_spot",
            "BrownSpot": "brown_spot",
            "False-Smut": "false_smut",
            "Healthy Rice Leaf": "healthy",
            "Healthy Rice beads": "healthy",
            "Healthy": "healthy",
            "healthy": "healthy",
            "Leaf-roller": "leaf_roller",
            "Leaf Scald": "leaf_scald",
            "Leaf scald": "leaf_scald",
            "Narrow brown": "narrow_brown",
            "Sheath Blight": "sheath_blight",
            "Rice-Tungro": "tungro",
        }
    )
    class_names: List[str] = field(
        default_factory=lambda: [
            "bacterial_leaf_blight",
            "bacterial_panicle_blight",
            "blast",
            "brown_spot",
            "false_smut",
            "healthy",
            "leaf_roller",
            "leaf_scald",
            "narrow_brown",
            "sheath_blight",
            "tungro",
        ]
    )

    # --- Hyperparameter training YOLO26s-P2 (From Scratch) ---
    model_yaml: str = "yolo26s-p2.yaml"
    train_epochs: int = 100  # lihat estimasi waktu yang dicetak setelah epoch ke-3 (batas sesi Kaggle 12 jam)
    train_batch: int = 16  # nbs=64 -> akumulasi gradien x4 => batch efektif 64 (sama dengan baseline)
    train_imgsz: int = 640
    train_workers: int = 4  # dikunci eksplisit agar augmentasi tidak bergantung pada jumlah CPU
    train_save_period: int = 10
    train_pretrained: bool = False
    train_amp: bool = True
    train_optimizer: str = "AdamW"
    train_lr0: float = 0.001  # default Ultralytics (0.01) terlalu besar untuk AdamW
    train_weight_decay: float = 0.01
    train_warmup_bias_lr: float = 0.0  # default 0.1 terlalu besar untuk Adam/AdamW
    train_hsv_h: float = 0.01
    train_hsv_s: float = 0.3
    train_hsv_v: float = 0.3
    train_translate: float = 0.2
    train_shear: float = 0.0
    train_perspective: float = 0.0
    train_scale: float = 0.4
    train_degrees: float = 0.0
    train_fliplr: float = 0.3
    train_flipud: float = 0.3
    train_mosaic: float = 0.4
    train_close_mosaic: int = 10
    train_erasing: float = 0.1
    train_auto_augment: Optional[str] = None
    train_cos_lr: bool = True
    train_patience: int = 0  # 0 = tanpa early stopping (seluruh epoch dijalankan)
    train_device: Any = field(default_factory=_pilih_device)
    train_project: Path = field(
        default_factory=lambda: (
            Path(r"/kaggle/working/train")
            if Path("/kaggle/working").exists()
            else Path.cwd() / "train"
        )
    )
    train_name: str = "train_yolo26s_p2_bifpn"

    # Diisi otomatis setelah training selesai (jalur pasti best.pt dari trainer)
    best_override: Optional[str] = None

    @property
    def class_to_id(self) -> Dict[str, int]:
        return {name: i for i, name in enumerate(self.class_names)}

    @property
    def best_weights_path(self) -> str:
        if self.best_override and Path(self.best_override).exists():
            return self.best_override
        candidates = [
            Path(self.train_project) / self.train_name / "weights" / "best.pt",
            Path("/kaggle/working/runs/detect/kaggle/working/train") / self.train_name / "weights" / "best.pt",
            Path.cwd() / "runs" / "detect" / self.train_name / "weights" / "best.pt",
        ]
        for c in candidates:
            if c.exists():
                return str(c)
        return str(candidates[0])

CFG = Config()
set_seed(CFG.seed)


## A.3 Cek Ketersediaan Folder & File Anotasi COCO
Memverifikasi keberadaan split dataset (train, valid, test) dan file anotasi `_annotations.coco.json`.


In [ ]:
def check_dataset_structure(cfg: Config) -> None:
    for split in cfg.coco_splits:
        split_dir = cfg.dataset_root / split
        json_path = split_dir / cfg.coco_annotation_file

        print(f"\n[{split}]")
        print("Folder :", split_dir)
        print("JSON   :", json_path)
        print("JSON exists :", json_path.exists())


check_dataset_structure(CFG)

## A.4 Audit Anotasi COCO Dataset
Menginspeksi validitas bounding box, mendeteksi bbox di luar resolusi citra, serta menghitung frekuensi kategori mentah.


In [ ]:
def audit_coco_json(json_path: Path) -> dict:
    data = json.loads(json_path.read_text(encoding="utf-8"))

    images = {img["id"]: img for img in data["images"]}
    categories = {cat["id"]: cat["name"] for cat in data["categories"]}

    anotasi_per_image = {}
    for anotasi in data["annotations"]:
        anotasi_per_image.setdefault(anotasi["image_id"], []).append(anotasi)

    gambar_tanpa_anotasi = [
        gambar
        for id_gambar, gambar in images.items()
        if id_gambar not in anotasi_per_image
    ]

    invalid_bbox = []
    for anotasi in data["annotations"]:
        if anotasi["image_id"] not in images:
            continue

        img = images[anotasi["image_id"]]
        W = img["width"]
        H = img["height"]
        bbox = anotasi.get("bbox")

        if bbox is None or len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
            invalid_bbox.append(anotasi)
            continue

        x, y, w, h = bbox
        if x < 0 or y < 0 or x + w > W or y + h > H:
            invalid_bbox.append(anotasi)

    class_counts = Counter()
    for anotasi in data["annotations"]:
        nama = categories.get(anotasi["category_id"], "UNKNOWN")
        class_counts[nama] += 1

    print(f"Gambar              : {len(images)}")
    print(f"Anotasi            : {len(data['annotations'])}")
    print(f"Gambar tanpa anotasi : {len(gambar_tanpa_anotasi)}")
    print(f"Bbox invalid/outside    : {len(invalid_bbox)}")
    print("Distribusi kategori:")
    for name, count in class_counts.most_common():
        print(f"{name:<35}: {count}")

    return data


def run_coco_audit(cfg: Config) -> Dict[str, dict]:
    data_coco = {}
    for split in cfg.coco_splits:
        json_path = cfg.dataset_root / split / cfg.coco_annotation_file
        print(f"AUDIT {split.upper()}")
        data_coco[split] = audit_coco_json(json_path)
    return data_coco


data_coco = run_coco_audit(CFG)

## A.5 Pembuatan Struktur Direktori YOLO
Membuat hierarki folder `images/{train,val,test}` dan `labels/{train,val,test}`.


In [ ]:
def create_yolo_dirs(cfg: Config) -> None:
    # Bersihkan keluaran lama agar eksekusi ulang identik dengan eksekusi pertama
    # (tanpa sisa label/cache). Hanya images/ dan labels/ yang dihapus, folder hasil training tidak disentuh.
    for sub in ("images", "labels"):
        shutil.rmtree(cfg.yolo_out / sub, ignore_errors=True)

    for split in ["train", "val", "test"]:
        (cfg.yolo_out / "images" / split).mkdir(parents=True, exist_ok=True)
        (cfg.yolo_out / "labels" / split).mkdir(parents=True, exist_ok=True)

    print("Folder YOLO berhasil dibuat.")


create_yolo_dirs(CFG)


## A.6 Konversi COCO ke Format YOLO & Pemetaan 11 Kelas Kanonikal
Normalisasi koordinat bbox `(xc, yc, w, h)`, pembersihan bbox invalid, serta standarisasi 18 label mentah ke 11 kelas resmi.


In [ ]:
def convert_coco_to_yolo(cfg: Config) -> None:
    class_to_id = cfg.class_to_id

    for split in cfg.coco_splits:
        input_dir = cfg.dataset_root / split
        json_path = input_dir / cfg.coco_annotation_file

        output_split = "val" if split == "valid" else split
        image_out = cfg.yolo_out / "images" / output_split
        label_out = cfg.yolo_out / "labels" / output_split

        with open(json_path, "r", encoding="utf-8") as f:
            coco = json.load(f)

        images = {image["id"]: image for image in coco["images"]}
        categories = {
            category["id"]: category["name"] for category in coco["categories"]
        }

        annotations_per_image = {image_id: [] for image_id in images}

        # dipakai untuk melacak kelas apa saja yang ada di gambar yang akhirnya di-skip
        raw_categories_per_image = {image_id: [] for image_id in images}

        invalid_count = 0
        clipped_count = 0
        unmapped_count = 0

        invalid_class_counts = Counter()
        clipped_class_counts = Counter()
        unmapped_class_counts = Counter()

        for ann in coco["annotations"]:
            image_id = ann["image_id"]
            category_name = categories[ann["category_id"]]
            raw_categories_per_image[image_id].append(category_name)

            mapped_class = cfg.class_mapping.get(category_name)

            if mapped_class is None:
                unmapped_count += 1
                unmapped_class_counts[category_name] += 1
                continue

            class_id = class_to_id[mapped_class]

            x, y, w, h = ann["bbox"]
            img_w = images[image_id]["width"]
            img_h = images[image_id]["height"]

            x1, y1, x2, y2 = x, y, x + w, y + h

            new_x1 = max(0, x1)
            new_y1 = max(0, y1)
            new_x2 = min(img_w, x2)
            new_y2 = min(img_h, y2)

            new_w = new_x2 - new_x1
            new_h = new_y2 - new_y1

            if new_w <= 0 or new_h <= 0:
                invalid_count += 1
                invalid_class_counts[category_name] += 1
                continue

            if new_x1 != x1 or new_y1 != y1 or new_x2 != x2 or new_y2 != y2:
                clipped_count += 1
                clipped_class_counts[category_name] += 1

            x_center = (new_x1 + new_x2) / 2 / img_w
            y_center = (new_y1 + new_y2) / 2 / img_h
            width = new_w / img_w
            height = new_h / img_h

            annotations_per_image[image_id].append(
                f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
            )

        # counter untuk gambar yang dimasukkan vs di-skip
        included_images = 0
        skipped_images = 0
        skipped_no_annotation = 0
        skipped_all_filtered = 0
        skipped_class_counts = Counter()

        for image_id, image_info in images.items():
            if not annotations_per_image[image_id]:
                skipped_images += 1

                raw_classes = raw_categories_per_image[image_id]
                if not raw_classes:
                    skipped_no_annotation += 1
                else:
                    skipped_all_filtered += 1
                    for c in raw_classes:
                        skipped_class_counts[c] += 1
                continue

            file_name = image_info["file_name"]
            source_image = input_dir / file_name

            if not source_image.exists():
                continue

            shutil.copy2(source_image, image_out / Path(file_name).name)

            label_name = Path(file_name).stem + ".txt"
            label_path = label_out / label_name

            with open(label_path, "w") as f:
                for line in annotations_per_image[image_id]:
                    f.write(line + "\n")

            included_images += 1

        # ================= RINGKASAN =================
        print(f"\n{'='*50}")
        print(f"  SPLIT: {split.upper()}")
        print(f"{'='*50}")

        print(f"\nTotal gambar                : {len(images)}")
        print(f"Gambar dimasukkan ke YOLO   : {included_images}")
        print(f"Gambar di-skip (total)      : {skipped_images}")
        print(f"  - tanpa annotation sama sekali : {skipped_no_annotation}")
        print(f"  - semua annotation dibuang     : {skipped_all_filtered}")

        print(f"\nBbox clipped   : {clipped_count}")
        print(f"Bbox invalid   : {invalid_count}")
        print(f"Bbox unmapped  : {unmapped_count}")

        print("\n-- Kelas pada gambar yang di-skip --")
        if skipped_class_counts:
            for name, count in skipped_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada / semua gambar skip memang tanpa annotation)")

        print("\n-- Kelas & jumlah bbox invalid --")
        if invalid_class_counts:
            for name, count in invalid_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada)")

        print("\n-- Kelas & jumlah bbox clipped --")
        if clipped_class_counts:
            for name, count in clipped_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada)")

        print("\n-- Kelas & jumlah bbox unmapped (tidak ada di CLASS_MAPPING) --")
        if unmapped_class_counts:
            for name, count in unmapped_class_counts.most_common():
                print(f"  {name:<35}: {count}")
        else:
            print("  (tidak ada)")


convert_coco_to_yolo(CFG)

## A.7 Pembuatan File Konfigurasi `data.yaml`
Menulis konfigurasi `data.yaml` yang menghubungkan Ultralytics ke direktori dataset hasil konversi.


In [ ]:
def write_data_yaml(cfg: Config) -> str:
    yaml_text = f"""
path: {cfg.yolo_out}

train: images/train
val: images/val
test: images/test

nc: {len(cfg.class_names)}

names:
"""
    for i, name in enumerate(cfg.class_names):
        yaml_text += f"  {i}: {name}\n"

    with open(cfg.yolo_out / "data.yaml", "w") as f:
        f.write(yaml_text)

    print(yaml_text)
    return yaml_text


write_data_yaml(CFG)

## A.8 Verifikasi Sinkronisasi Citra & Label
Memvalidasi keselarasan jumlah file citra dan file label pada seluruh split dataset.


In [ ]:
def verify_yolo_dataset(cfg: Config) -> None:
    for split in ["train", "val", "test"]:
        images = list((cfg.yolo_out / "images" / split).glob("*"))
        labels = list((cfg.yolo_out / "labels" / split).glob("*.txt"))
        print(split, "images:", len(images), "labels:", len(labels))


def show_sample_label(cfg: Config, split: str = "train") -> None:
    label_files = list((cfg.yolo_out / "labels" / split).glob("*.txt"))
    print(label_files[0])
    print()
    print(label_files[0].read_text())


verify_yolo_dataset(CFG)
show_sample_label(CFG, split="train")

## A.9 Visualisasi Sampel Ground Truth
Menampilkan sampel citra dan bounding box untuk konfirmasi visual hasil pra-pemrosesan.


In [ ]:
def visualisasi_sampel(
    output_dir: Path, split: str = "train", sample_size: int = 20, cfg: Config = CFG
) -> None:
    images_dir = output_dir / "images" / split
    labels_dir = output_dir / "labels" / split

    image_files = [
        p
        for p in images_dir.rglob("*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    random.seed(cfg.seed)

    samples = random.sample(image_files, min(sample_size, len(image_files)))

    cols = 3
    rows = int(np.ceil(len(samples) / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))

    axes = np.array(axes).reshape(-1)

    for ax, image_path in zip(axes, samples):
        image = Image.open(image_path)

        W, H = image.size

        ax.imshow(image)

        label_path = (labels_dir / image_path.relative_to(images_dir)).with_suffix(
            ".txt"
        )

        # Kalau label tidak ada
        if not label_path.exists():
            ax.set_title(image_path.name)
            ax.axis("off")
            continue

        lines = label_path.read_text(encoding="utf-8").splitlines()

        for line in lines:
            if not line.strip():
                continue

            class_id, xc, yc, w, h = map(float, line.split())

            class_id = int(class_id)

            # YOLO normalized -> pixel
            xc *= W
            yc *= H
            w *= W
            h *= H

            x1 = xc - w / 2
            y1 = yc - h / 2

            rect = plt.Rectangle((x1, y1), w, h, fill=False, linewidth=2)

            ax.add_patch(rect)

            ax.text(
                x1, y1, cfg.class_names[class_id], fontsize=8, backgroundcolor="white"
            )

        ax.set_title(image_path.name)

        ax.axis("off")

    for ax in axes[len(samples) :]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


visualisasi_sampel(CFG.yolo_out, split="train", sample_size=20)

## A.10 Analisis Distribusi Kelas
Menghitung dan memetakan jumlah instans per kelas pada train, val, dan test untuk mendeteksi *class imbalance*.


In [ ]:
def compute_class_distribution(cfg: Config):
    import pandas as pd

    distribution = {}

    for split in ["train", "val", "test"]:
        label_dir = cfg.yolo_out / "labels" / split

        counter = Counter()

        for label_path in label_dir.glob("*.txt"):
            lines = label_path.read_text(encoding="utf-8").splitlines()

            for line in lines:
                if not line.strip():
                    continue

                class_id = int(line.split()[0])

                counter[class_id] += 1

        distribution[split] = counter

    df = pd.DataFrame(
        {
            "class": cfg.class_names,
            "train": [distribution["train"][i] for i in range(len(cfg.class_names))],
            "val": [distribution["val"][i] for i in range(len(cfg.class_names))],
            "test": [distribution["test"][i] for i in range(len(cfg.class_names))],
        }
    )

    df["total"] = df["train"] + df["val"] + df["test"]

    df = df.sort_values("total", ascending=False).reset_index(drop=True)

    print(df)
    return df


class_distribution_df = compute_class_distribution(CFG)

## A.11 Diagnostik Duplikasi Antar-Split (Data Leakage)
Dataset berasal dari ekspor Roboflow: satu gambar sumber dapat muncul sebagai beberapa salinan hasil augmentasi
(nama berpola `<sumber>_jpg.rf.<hash>.jpg`). Sel ini hanya **melaporkan** apakah gambar sumber yang sama muncul di
lebih dari satu split (berdasarkan nama sumber dan hash isi file). Split resmi panitia tetap dipakai apa adanya,
tidak ada data yang dibuang atau dipindah.

In [ ]:
import hashlib
import re


def _nama_sumber(file_name: str) -> str:
    """Buang akhiran Roboflow `_jpg.rf.<hash>` agar salinan augmentasi dari satu sumber dikenali."""
    return re.split(r"_jpg\.rf\.|\.rf\.", Path(file_name).name)[0]


def cek_kebocoran_antar_split(cfg: Config) -> None:
    sumber: Dict[str, set] = {}
    hash_isi: Dict[str, set] = {}

    for split in cfg.coco_splits:
        json_path = cfg.dataset_root / split / cfg.coco_annotation_file
        coco = json.loads(json_path.read_text(encoding="utf-8"))
        nama = [im["file_name"] for im in coco["images"]]
        sumber[split] = {_nama_sumber(n) for n in nama}

        hashes = set()
        for n in nama:
            p = cfg.dataset_root / split / n
            if p.exists():
                hashes.add(hashlib.md5(p.read_bytes()).hexdigest())
        hash_isi[split] = hashes
        print(f"{split:<6}: {len(nama):>6} gambar | {len(sumber[split]):>6} sumber unik")

    print("\nIrisan antar-split:")
    pasangan = [("train", "valid"), ("train", "test"), ("valid", "test")]
    for a, b in pasangan:
        irisan_nama = sumber[a] & sumber[b]
        irisan_hash = hash_isi[a] & hash_isi[b]
        print(f"  {a:>5} & {b:<5}: sumber sama = {len(irisan_nama):>5} | isi file identik = {len(irisan_hash):>5}")


cek_kebocoran_antar_split(CFG)


---
### Bagian B — Pelatihan Model (From Scratch)
Inisialisasi model YOLO dari konfigurasi YAML murni tanpa bobot pra-latih (*transfer learning dilarang*).


### Bagian B — Arsitektur & Pelatihan (From Scratch)

## B.0 Arsitektur: YOLO26s dengan Head P2 (dari Konfigurasi YAML)
Model dibangun dari **arsitektur kosong** lewat file konfigurasi YAML (tanpa bobot pra-latih). Konfigurasi di bawah
menambahkan jalur fitur stride-4 (P2, 160x160 pada input 640) ke neck dan head deteksi, agar lesi mikro seperti
`brown_spot` (median kurang dari 0,3% luas citra) tidak hilang saat downsampling. File ditulis dari dalam notebook
sehingga arsitektur terlihat utuh di sini dan tidak bergantung pada file di luar notebook.

In [ ]:
YOLO26S_P2_YAML = r'''# YOLO26s dengan head deteksi P2/4 - P5/32 (salinan konfigurasi Ultralytics yolo26-p2.yaml, nc=11)
# Diinisialisasi dari arsitektur kosong: TIDAK ada bobot pra-latih.
# Ultralytics YOLO26 object detection model with P2/4 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo26
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 11 # number of classes
end2end: True # whether to use end-to-end mode
reg_max: 1 # DFL bins
scales: # model compound scaling constants, i.e. 'model=yolo26n-p2.yaml' will call yolo26-p2.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 329 layers, 2,662,400 parameters, 2,662,400 gradients, 9.5 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 329 layers, 9,765,856 parameters, 9,765,856 gradients, 27.8 GFLOPs
  m: [0.50, 1.00, 512] # summary: 349 layers, 21,144,288 parameters, 21,144,288 gradients, 91.4 GFLOPs
  l: [1.00, 1.00, 512] # summary: 489 layers, 25,815,520 parameters, 25,815,520 gradients, 115.3 GFLOPs
  x: [1.00, 1.50, 512] # summary: 489 layers, 57,935,232 parameters, 57,935,232 gradients, 256.9 GFLOPs

# YOLO26n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

# YOLO26n head
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16 (P3/8-small)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]] # cat backbone P2
  - [-1, 2, C3k2, [128, True]] # 19 (P2/4-xsmall)

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 16], 1, Concat, [1]] # cat head P3
  - [-1, 2, C3k2, [256, True]] # 22 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, True]] # 25 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 28 (P5/32-large)

  - [[19, 22, 25, 28], 1, Detect, [nc]] # Detect(P2, P3, P4, P5)
'''

Path(CFG.model_yaml).write_text(YOLO26S_P2_YAML, encoding="utf-8")
print("Arsitektur ditulis ke:", Path(CFG.model_yaml).resolve())


## B.1 Modul Kustom: BiFPN Fast Normalized Fusion & Loss EIoU
1. **`BiFPN_Concat`**: fusi fitur berbobot terpelajari $w_i \ge 0$ pada setiap `Concat` di head:
   $$O = \text{Concat}\left(\frac{w_1}{\epsilon + w_1 + w_2} \cdot x_1, \frac{w_2}{\epsilon + w_1 + w_2} \cdot x_2\right)$$
2. **EIoU** menggantikan CIoU di `BboxLoss`; penalti lebar dan tinggi dipisah agar lesi tipis memanjang
   (`bacterial_leaf_blight`) tidak terdistorsi gradiennya:
   $$\mathcal{L}_{EIoU} = 1 - IoU + \frac{\rho^2(b, b^{gt})}{c^2} + \frac{\rho^2(w, w^{gt})}{C_w^2} + \frac{\rho^2(h, h^{gt})}{C_h^2}$$
3. **Lewati pengecekan AMP bawaan**: sebelum training, Ultralytics menjalankan `check_amp` yang mengunduh `yolo26n.pt`
   hanya untuk membandingkan keluaran FP16 dan FP32. Bobot itu **tidak pernah dipakai untuk melatih**, tetapi unduhannya
   tampak di log dan tidak sesuai semangat aturan "tanpa bobot pra-latih". Pengecekan ini dilewati (AMP tetap aktif di GPU)
   sehingga tidak ada bobot bawaan yang diunduh.

> Patch dipasang di dalam proses notebook. Karena itu training memakai **satu GPU** (lihat `_pilih_device`).

In [ ]:
import torch
import torch.nn as nn
import ultralytics
import ultralytics.nn.modules as modules
import ultralytics.nn.tasks as tasks
import ultralytics.utils.loss as loss_module
import ultralytics.engine.trainer as trainer_module

EXPECTED_ULTRALYTICS = "8.4.163"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
if ultralytics.__version__ != EXPECTED_ULTRALYTICS:
    print(
        f"PERINGATAN: kode ini diuji pada ultralytics=={EXPECTED_ULTRALYTICS}; versi lain dapat "
        "mengubah nama internal yang di-patch dan membuat hasil audit berbeda."
    )


class BiFPN_Concat(modules.Concat):
    """BiFPN Fast Normalized Feature Fusion untuk dua masukan."""

    def __init__(self, dimension=1):
        super().__init__(dimension=dimension)
        self.w = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.epsilon = 1e-4

    def forward(self, x):
        if len(x) == 2:
            w = torch.relu(self.w)
            norm_w = w / (torch.sum(w) + self.epsilon)
            return torch.cat([norm_w[0] * x[0], norm_w[1] * x[1]], self.d)
        return super().forward(x)


def bbox_iou_eiou(box1, box2, xywh=True, GIoU=False, DIoU=False, CIoU=False, eps=1e-7):
    """Mengembalikan `IoU - penalti EIoU` (loss = 1 - nilai ini), kompatibel dengan ultralytics.utils.metrics.bbox_iou."""
    if xywh:
        (x1, y1, w1, h1), (x2, y2, w2, h2) = box1.chunk(4, -1), box2.chunk(4, -1)
        w1_, h1_, w2_, h2_ = w1 / 2, h1 / 2, w2 / 2, h2 / 2
        b1_x1, b1_x2, b1_y1, b1_y2 = x1 - w1_, x1 + w1_, y1 - h1_, y1 + h1_
        b2_x1, b2_x2, b2_y1, b2_y2 = x2 - w2_, x2 + w2_, y2 - h2_, y2 + h2_
    else:
        b1_x1, b1_y1, b1_x2, b1_y2 = box1.chunk(4, -1)
        b2_x1, b2_y1, b2_x2, b2_y2 = box2.chunk(4, -1)
        w1, h1 = b1_x2 - b1_x1, b1_y2 - b1_y1 + eps
        w2, h2 = b2_x2 - b2_x1, b2_y2 - b2_y1 + eps

    inter = (b1_x2.minimum(b2_x2) - b1_x1.maximum(b2_x1)).clamp_(0) * (
        b1_y2.minimum(b2_y2) - b1_y1.maximum(b2_y1)
    ).clamp_(0)
    union = w1 * h1 + w2 * h2 - inter + eps
    iou = inter / union

    cw = b1_x2.maximum(b2_x2) - b1_x1.minimum(b2_x1)
    ch = b1_y2.maximum(b2_y2) - b1_y1.minimum(b2_y1)
    c2 = cw.pow(2) + ch.pow(2) + eps
    rho2 = ((b2_x1 + b2_x2 - b1_x1 - b1_x2).pow(2) + (b2_y1 + b2_y2 - b1_y1 - b1_y2).pow(2)) / 4
    rho_w2 = (w2 - w1).pow(2)
    rho_h2 = (h2 - h1).pow(2)

    return iou - (rho2 / c2 + rho_w2 / (cw.pow(2) + eps) + rho_h2 / (ch.pow(2) + eps))


def _check_amp_tanpa_unduhan(model) -> bool:
    """Pengganti `check_amp`: tetap menghormati perangkat (AMP mati di CPU/MPS) tetapi tanpa mengunduh bobot."""
    return next(model.parameters()).device.type not in {"cpu", "mps"}


def terapkan_patch_kustom() -> None:
    """Pasang modul kustom ke mesin Ultralytics (idempoten; aman dipanggil berulang)."""
    assert hasattr(loss_module, "bbox_iou") and hasattr(tasks, "Concat") and hasattr(trainer_module, "check_amp"), (
        "Struktur internal Ultralytics berbeda dari 8.4.163; patch tidak dapat dipasang."
    )
    tasks.Concat = BiFPN_Concat  # parse_model membaca nama global ini saat membangun head
    loss_module.bbox_iou = bbox_iou_eiou  # dipanggil oleh BboxLoss.forward
    trainer_module.check_amp = _check_amp_tanpa_unduhan
    assert tasks.Concat is BiFPN_Concat and loss_module.bbox_iou is bbox_iou_eiou


terapkan_patch_kustom()
print("BiFPN_Concat, EIoU, dan pelewatan check_amp terpasang.")


## B.2 Trainer Kustom: Inisialisasi Kaiming Normal & Pagar Pengaman From-Scratch
Ultralytics membangun modelnya sendiri di dalam trainer dari nama file YAML, sehingga model yang dibuat dan
diinisialisasi di luar trainer **tidak pernah dipakai**. Karena itu inisialisasi dipasang di dalam `ScratchTrainer.get_model()`,
tepat setelah model dibangun dan sebelum optimizer dibuat.

- Kaiming Normal (`fan_out`, `relu`) diterapkan ke semua `Conv2d` **kecuali head `Detect`**. Head dikecualikan karena
  `Detect.bias_init()` menetapkan prior bias kelas dan box; menolkannya akan memperlambat konvergensi awal.
- `get_model()` menolak bobot apa pun yang dimuat (kecuali melanjutkan checkpoint buatan sendiri saat `resume`),
  sehingga pelatihan tidak mungkin tanpa sengaja memakai transfer learning.
- `get_model()` memastikan ada tepat **6 modul `BiFPN_Concat`** pada model; bila patch tidak aktif, training berhenti
  seketika dengan pesan jelas, alih-alih berjalan diam-diam tanpa BiFPN.

In [ ]:
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules import Detect

N_BIFPN_DIHARAPKAN = 6  # jumlah Concat pada head yolo26-p2


def init_kaiming_weights(model) -> int:
    """Kaiming Normal pada seluruh Conv2d kecuali head Detect. Mengembalikan jumlah lapisan yang diinisialisasi."""
    param_head = {id(p) for m in model.modules() if isinstance(m, Detect) for p in m.parameters()}
    n = 0
    for m in model.modules():
        if isinstance(m, nn.Conv2d) and id(m.weight) not in param_head:
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
            n += 1
    return n


class ScratchTrainer(DetectionTrainer):
    """DetectionTrainer yang memastikan model dilatih dari nol dengan modul kustom aktif."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        if weights is not None and not getattr(self, "resume", False):
            raise RuntimeError("Bobot pra-latih terdeteksi: dilarang oleh aturan lomba (wajib from scratch).")
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)
        if weights is None:
            n_init = init_kaiming_weights(model)
            print(f"[INIT] Kaiming Normal diterapkan pada {n_init} lapisan Conv2d (head Detect dikecualikan).")
        n_bifpn = sum(isinstance(m, BiFPN_Concat) for m in model.modules())
        if n_bifpn != N_BIFPN_DIHARAPKAN:
            raise RuntimeError(
                f"BiFPN_Concat aktif pada {n_bifpn} modul (seharusnya {N_BIFPN_DIHARAPKAN}). "
                "Pastikan sel B.1 sudah dijalankan dan training memakai satu GPU."
            )
        if loss_module.bbox_iou is not bbox_iou_eiou:
            raise RuntimeError("Loss EIoU tidak aktif (bbox_iou belum di-patch).")
        print(f"[CEK] BiFPN_Concat = {n_bifpn} modul | loss = EIoU | bobot pra-latih = tidak ada")
        return model


def _estimasi_waktu(trainer) -> None:
    """Setelah epoch ke-3, cetak estimasi total waktu agar batas sesi Kaggle (12 jam) bisa diantisipasi dini."""
    if trainer.epoch - trainer.start_epoch + 1 == 3:
        rata2 = (time.time() - trainer.train_time_start) / 3
        total_jam = rata2 * trainer.epochs / 3600
        print(f"[WAKTU] ~{rata2 / 60:.1f} menit/epoch -> estimasi {total_jam:.1f} jam untuk {trainer.epochs} epoch.")
        if total_jam > 11:
            print("[WAKTU] PERINGATAN: melebihi ~11 jam. Hentikan sekarang dan turunkan `train_epochs` di Config.")


## B.3 Pelatihan Model (From Scratch, GPU Tunggal)
Optimizer **AdamW** dengan `lr0=0.001`, `weight_decay=0.01`, `warmup_bias_lr=0.0`, dijadwalkan cosine. Nilai ini
ditetapkan eksplisit karena bawaan Ultralytics (`lr0=0.01`, `warmup_bias_lr=0.1`) ditujukan untuk SGD dan terlalu besar
untuk AdamW. Semua seed dikunci (`seed=42`, `deterministic=True`) dan `workers` ditetapkan tetap.

Set `RUN_TRAINING = False` hanya bila melanjutkan evaluasi dari `best.pt` buatan sendiri yang sudah ada;
untuk Audit Run, biarkan `True` agar seluruh alur berjalan dari data mentah sampai evaluasi.

In [ ]:
from ultralytics import YOLO


def train_model(cfg: Config):
    dev = cfg.train_device
    if (isinstance(dev, (list, tuple)) and len(dev) > 1) or (isinstance(dev, str) and "," in dev):
        raise RuntimeError(
            "Multi-GPU (DDP) tidak didukung: patch BiFPN/EIoU hanya berlaku di proses notebook. "
            "Gunakan satu GPU (train_device=0)."
        )

    terapkan_patch_kustom()  # pastikan patch aktif di proses ini

    # 1. Arsitektur kosong dari YAML (tanpa bobot pra-latih)
    model = YOLO(cfg.model_yaml)
    model.add_callback("on_fit_epoch_end", _estimasi_waktu)

    # 2. Pelatihan dengan trainer kustom (Kaiming init + pagar pengaman) -- AdamW + cosine
    results = model.train(
        trainer=ScratchTrainer,
        data=str(cfg.yolo_out / "data.yaml"),
        epochs=cfg.train_epochs,
        batch=cfg.train_batch,
        imgsz=cfg.train_imgsz,
        workers=cfg.train_workers,
        save_period=cfg.train_save_period,
        pretrained=cfg.train_pretrained,
        seed=cfg.seed,
        deterministic=True,
        amp=cfg.train_amp,
        optimizer=cfg.train_optimizer,
        lr0=cfg.train_lr0,
        weight_decay=cfg.train_weight_decay,
        warmup_bias_lr=cfg.train_warmup_bias_lr,
        cos_lr=cfg.train_cos_lr,
        patience=cfg.train_patience,
        # HSV Augmentation
        hsv_h=cfg.train_hsv_h,
        hsv_s=cfg.train_hsv_s,
        hsv_v=cfg.train_hsv_v,
        # Geometric Augmentation
        translate=cfg.train_translate,
        shear=cfg.train_shear,
        perspective=cfg.train_perspective,
        scale=cfg.train_scale,
        degrees=cfg.train_degrees,
        # Flip & Mosaic
        fliplr=cfg.train_fliplr,
        flipud=cfg.train_flipud,
        mosaic=cfg.train_mosaic,
        close_mosaic=cfg.train_close_mosaic,
        erasing=cfg.train_erasing,
        auto_augment=cfg.train_auto_augment,
        project=str(cfg.train_project),
        name=cfg.train_name,
        device=cfg.train_device,
        exist_ok=True,
    )

    # 3. Simpan jalur pasti best.pt dari trainer untuk tahap evaluasi
    best = Path(model.trainer.best)
    cfg.best_override = str(best if best.exists() else model.trainer.last)
    print("Bobot terbaik:", cfg.best_override)
    return model, results


RUN_TRAINING = True
if RUN_TRAINING:
    model_trained, results_trained = train_model(CFG)


---
### Bagian C — Evaluasi Model & Verifikasi Metrik
Evaluasi komprehensif pada set validasi dan test set menggunakan metrik mAP@50 dan F1-Score.


## C.1 Memuat Bobot Model Terbaik (`best.pt`)


In [ ]:
def load_best_model(weights_path: str) -> YOLO:
    return YOLO(weights_path)


best_model = load_best_model(CFG.best_weights_path)

## C.2 Evaluasi pada Data Validasi (Validation Set)


In [ ]:
def evaluate_model(model: YOLO, cfg: Config, split: str):
    return model.val(
        data=str(cfg.yolo_out / "data.yaml"),
        split=split,
        imgsz=cfg.train_imgsz,
        batch=16,
        workers=cfg.train_workers,
        device=cfg.train_device,
    )


def print_per_class(metrics, names: List[str]) -> None:
    """Tabel metrik per kelas: kelas lemah (mis. brown_spot) langsung terlihat."""
    print(f"{'kelas':<26}{'P':>7}{'R':>7}{'mAP50':>8}{'mAP50-95':>10}")
    for i, c in enumerate(metrics.box.ap_class_index):
        p, r, ap50, ap = metrics.box.class_result(i)
        print(f"{names[int(c)]:<26}{p:>7.3f}{r:>7.3f}{ap50:>8.3f}{ap:>10.3f}")


metrics = evaluate_model(best_model, CFG, split="val")

print("mAP50 :", metrics.box.map50)
print("mAP50-95 :", metrics.box.map)
print()
print_per_class(metrics, CFG.class_names)


## C.3 Evaluasi Final pada Data Uji (Test Set) & Kalkulasi F1-Score


In [ ]:
def compute_f1(precision: float, recall: float) -> float:
    if precision + recall > 0:
        return 2 * precision * recall / (precision + recall)
    return 0.0

final_metrics = evaluate_model(best_model, CFG, split="test")

print("=== EVALUASI RESMI DEWAN JURI (TEST SET) ===")
print(f"Test mAP@50    : {final_metrics.box.map50:.4f}")
print(f"Test mAP@50-95 : {final_metrics.box.map:.4f}")

test_precision = final_metrics.box.mp
test_recall = final_metrics.box.mr
test_f1 = compute_f1(test_precision, test_recall)

print(f"Test Precision : {test_precision:.4f}")
print(f"Test Recall    : {test_recall:.4f}")
print(f"Test F1-Score  : {test_f1:.4f}")

# Ekstraksi F1 optimal dari kurva F1 resmi Ultralytics
try:
    f1_curve = final_metrics.box.f1
    mean_f1 = f1_curve.mean(axis=0)
    opt_idx = int(np.argmax(mean_f1))
    opt_f1 = float(mean_f1[opt_idx])
    opt_conf = float(final_metrics.box.curves_results[0][0][opt_idx])
    print(f"Optimal F1 (at conf={opt_conf:.2f}) : {opt_f1:.4f}")
except Exception:
    pass

print()
print_per_class(final_metrics, CFG.class_names)


## C.4 Pemeriksaan Kepatuhan: Tidak Ada Bobot Bawaan Ultralytics
Memastikan tidak ada berkas bobot pra-latih (mis. `yolo26n.pt`) yang terunduh atau tertinggal di direktori kerja.
Berkas `best.pt`, `last.pt`, dan `epochN.pt` hasil pelatihan sendiri adalah wajar.

In [ ]:
def audit_bobot_pretrained(root: Path = Path.cwd()) -> None:
    from ultralytics.utils import WEIGHTS_DIR

    nama_bawaan = ("yolo26", "yolo11", "yolov8", "yolov5", "yolov10", "yolov9", "rtdetr", "sam")
    kandidat = list(root.rglob("*.pt")) + list(Path(WEIGHTS_DIR).glob("*.pt"))
    mencurigakan = sorted({str(p) for p in kandidat if p.name.lower().startswith(nama_bawaan)})
    if mencurigakan:
        print("PERINGATAN: berkas bobot bawaan ditemukan, hapus sebelum submit:")
        for p in mencurigakan:
            print("  -", p)
    else:
        print("OK: tidak ada berkas bobot bawaan Ultralytics yang terunduh atau tertinggal.")


audit_bobot_pretrained()
